<a href="https://colab.research.google.com/github/Omar-Elgendey/egyptian-arabic-llm-post-training/blob/main/notebooksfinetune_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
DRIVE_PROCESSED = "/content/drive/.shortcut-targets-by-id/1HdzENVbmRV6W1QLyilD4dohEr3bbGUxA/ds/data/processed"

print(DRIVE_PROCESSED)

/content/drive/.shortcut-targets-by-id/1HdzENVbmRV6W1QLyilD4dohEr3bbGUxA/ds/data/processed


In [ ]:
%cd /content/egyptian-arabic-llm-post-training

!git pull origin main

/content/egyptian-arabic-llm-post-training
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 365 bytes | 365.00 KiB/s, done.
From https://github.com/Omar-Elgendey/egyptian-arabic-llm-post-training
 * branch            main       -> FETCH_HEAD
   f2b8e4d..03f68de  main       -> origin/main
Updating f2b8e4d..03f68de
Fast-forward
 configs/egyptian_sft_qlora.yaml | 8 ++++----
 1 file changed, 4 insertions(+), 4 deletions(-)


In [ ]:
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
!cd LLaMA-Factory && pip install -e .

fatal: destination path 'LLaMA-Factory' already exists and is not an empty directory.
Obtaining file:///content/egyptian-arabic-llm-post-training/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llamafactory (pyproject.toml) ... done
  Created wheel for llamafactory: filename=llamafactory-0.9.6.dev0-py3-none-any.whl size=27467 sha256=b581d14193e078dc2cb579b7b44776cb19c6d04a5e1ab2733f6b35a29c91274e
  Stored in directory: /tmp/pip-ephem-wheel-cache-2wre21eb/wheels/96/c8/10/4b7dae6a58066d19bcd78a6ff618462df93e4b9871c93612c2
Successfully built llamafactory
  Attempting uninstall: llamafactory
    Found existing installation: llamafactory 0.9.6.dev0
    Uninstalling llamafactory-0.9.6.dev0:
      Successfully uninstalled llamafactory-0.9.6.dev0


In [ ]:
!pip install -q -U bitsandbytes

In [ ]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
!hf auth login --token {hf_token}

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `egyptian-arabic-translation-project` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `egyptian-arabic-translation-project`


In [ ]:
import json
import random

eval_path = f"{DRIVE_PROCESSED}/sft_eval.jsonl"
subset_path = "/content/LLaMA-Factory/data/sft_eval_500.jsonl"

random.seed(42)

with open(eval_path, "r", encoding="utf-8") as f:
    eval_data = [json.loads(line) for line in f]

eval_subset = random.sample(eval_data, 500)

with open(subset_path, "w", encoding="utf-8") as f:
    for sample in eval_subset:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

print(f"Created evaluation subset: {len(eval_subset)} examples")

Created evaluation subset: 500 examples


In [ ]:
import json
import random

train_path = f"{DRIVE_PROCESSED}/sft_train.jsonl"
subset_path = "/content/LLaMA-Factory/data/sft_train_50k.jsonl"

sample_size = 50_000
random.seed(42)

train_subset = []

with open(train_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        sample = json.loads(line)

        if i < sample_size:
            train_subset.append(sample)
        else:
            j = random.randint(0, i)

            if j < sample_size:
                train_subset[j] = sample

with open(subset_path, "w", encoding="utf-8") as f:
    for sample in train_subset:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

print(f"Created training subset: {len(train_subset)} examples")

Created training subset: 50000 examples


In [ ]:
import shutil
import json

project_config = "/content/egyptian-arabic-llm-post-training/configs/dataset_info.json"

llama_factory_config = "/content/LLaMA-Factory/data/dataset_info.json"

shutil.copy(project_config, llama_factory_config)

with open(llama_factory_config, "r", encoding="utf-8") as f:
    dataset_info = json.load(f)

dataset_info["egyptian_sft"]["file_name"] = "sft_train_50k.jsonl"
dataset_info["egyptian_sft_eval"]["file_name"] = "sft_eval_500.jsonl"

with open(llama_factory_config, "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

print("dataset_info.json configured for 50k training and 500 evaluation.")

dataset_info.json configured for 50k training and 500 evaluation.


In [ ]:
import shutil

shutil.copy(
    "/content/egyptian-arabic-llm-post-training/configs/egyptian_sft_qlora.yaml",
    "/content/LLaMA-Factory/examples/train_lora/egyptian_sft_qlora.yaml"
)

print("SFT config copied.")

SFT config copied.


In [ ]:

%cd /content/LLaMA-Factory

!llamafactory-cli train examples/train_lora/egyptian_sft_qlora.yaml

/content/LLaMA-Factory
/usr/local/lib/python3.13/dist-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[INFO|2026-10-07 16:40:41] llamafactory.hparams.parser:144 >> Resuming training from /content/drive/MyDrive/ds/models/egyptian-qwen-sft-50k/checkpoint-3000.
[INFO|2026-10-07 16:40:41] llamafactory.hparams.parser:144 >> Change `output_dir` or use `overwrite_output_dir` to avoid.
[INFO|2026-10-07 16:40:41] llamafactory.hparams.parser:651 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.float16
[INFO|configuration_utils.py:780] 2026-10-07 16:40:41,503 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf80

In [ ]:
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/ds/models/egyptian-qwen-sft-50k"
)

=checkpoints = [
    p for p in OUTPUT_DIR.glob("checkpoint-*")
    if p.is_dir()
]

latest_checkpoint = max(
    checkpoints,
    key=lambda p: int(p.name.split("-")[-1])
)

print("Using:", latest_checkpoint)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    str(latest_checkpoint)
)

model.eval()

print("SFT model loaded successfully.")

Using: /content/drive/MyDrive/ds/models/egyptian-qwen-sft-50k/checkpoint-3001


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

SFT model loaded successfully.


In [ ]:
test_prompts = [
    "إزيك؟ عامل إيه؟ كلمني بالمصري.",

    "ممكن تشرحلي يعني إيه RAG بطريقة بسيطة وبالمصري؟",

    "أنا عايز أتعلم Python، أبدأ منين؟",

    "إيه الفرق بين الـ fine-tuning والـ RAG؟",

    "ازاي أعمل list في Python؟ اديني مثال بسيط.",

    "اكتبلي function في Python بتحسب متوسط الأرقام.",

    "لو معايا 100 جنيه وصرفت 35 جنيه، فاضل معايا كام؟",

    "ترجم للإنجليزي: أنا رايح الجامعة بكرة.",

    "ezzayak? momken teshra7ly ya3ny eh RAG?",

    "ana 3ayez at3alem Python, abda2 ezay?"
]

for i, prompt in enumerate(test_prompts, 1):
    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    print("=" * 80)
    print(f"TEST {i}")
    print(f"Q: {prompt}")
    print(f"A: {answer}")

In [ ]:
import torch

test_prompts = [
    # Egyptian Arabic / MSA
    "ممكن تترجملي من الفصحى للمصري: «سوف أعود إلى المنزل غدًا.»",

    "ممكن تترجملي من المصري للفصحى: «بيتهيالي إن الموضوع ده مش هيخلص بسرعة.»",

    "اكتبلي فقرة قصيرة بالمصري عن أول يوم في الجامعة.",

    # Franco Egyptian
    "ana 3ayez a3raf ezay a3mel account 3ala GitHub",

    "momken t2oly eh el far2 ben AI w Machine Learning?",

    # English ↔ Egyptian
    "ترجملي للإنجليزي: أنا مشغول دلوقتي، ممكن نتكلم بعدين؟",

    "ترجملي للمصري: «You must be feeling better today.»",

    # Code
    "اكتبلي Python function بتاخد list من الأرقام وترجع أكبر رقم.",

    "استخدم NumPy عشان تعمل array أبعاده 3×3×3 وتملأه بأرقام عشوائية من -10 إلى 10.",

    # Math / reasoning
    "لو 3 أقلام بـ45 جنيه، يبقى 7 أقلام بكام؟",

    "لو 132 بالأساس k بيساوي 42 بالأساس 10، أوجد k.",

    # Instruction following
    "اكتب 3 أسباب لتعلم Python، وكل سبب في سطر واحد فقط.",

    # Explanation
    "اشرحلي الفرق بين supervised وunsupervised learning بالمصري وبمثال بسيط.",

    # Multi-turn-style
    "أنا بتعلم Python وعايز أبدأ بمشروع بسيط، تقترح إيه؟",

    # Strict output
    "احسب 25 × 16. اكتب الرقم النهائي فقط."
]


def generate_answer(prompt):
    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
        )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    return answer


results = []

for i, prompt in enumerate(test_prompts, 1):
    answer = generate_answer(prompt)

    result = (
        f"{'=' * 80}\n"
        f"TEST {i}\n"
        f"Q: {prompt}\n\n"
        f"A: {answer}\n"
    )

    print(result)
    results.append(result)


# Save results
results_path = "/content/sft_test_results.txt"

with open(results_path, "w", encoding="utf-8") as f:
    f.write("\n".join(results))

print(f"\nSaved results to: {results_path}")

In [ ]:
# base model

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

base_model.eval()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [ ]:
def generate_base(prompt, max_new_tokens=150):
    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(base_model.device)

    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

In [ ]:
base_results = []

for i, prompt in enumerate(test_prompts, 1):
    answer = generate_base(prompt)

    base_results.append({
        "id": i,
        "prompt": prompt,
        "answer": answer
    })

    print(f"\n{'='*60}")
    print(f"TEST {i}")
    print(f"Q: {prompt}")
    print(f"A: {answer}")


TEST 1
Q: ممكن تترجملي من الفصحى للمصري: «سوف أعود إلى المنزل غدًا.»
A: بالطبع، يمكنني الترجمة لـ "سوف أعود إلى المنزل غدًا" من اللغة العربية الفصحى إلى اللغة العربية المصغرة (العربية الأدبية):

سأعود إلى البيت غداً

هذا الترجمة تعبر عن نفس المعنى اللغوي للجملة الأصلية في اللغة العربية الفصحى.

TEST 2
Q: ممكن تترجملي من المصري للفصحى: «بيتهيالي إن الموضوع ده مش هيخلص بسرعة.»
A: بالطبع، يمكنني الترجمة للغة الفصحى:

"أنا أشعر أن الموضوع ليس سهل الحل في الوقت الحالي."

هذه الترجمة تعبر عن نفس المعنى في اللغة العربية الفصحى، حيث يشير الشخص إلى أنه لا يستطيع حل المشكلة بشكل سريع أو سهل.

TEST 3
Q: اكتبلي فقرة قصيرة بالمصري عن أول يوم في الجامعة.
A: أول يوم في الجامعة كان مليئًا بالحماس والرعب. كنت أشعر بcuriosity (الدهشة) لما سيأتي، وعندما دخلت إلى المبنى، رأيت الطلاب يتجهون نحو الأقسام المختلفة. كانت هناك صخب وضجيج، لكنني كنت متحمسًا للاستكشاف. بدأت بالبحث عن غرفتي الجديدة، ووجدت أنها تقع في الطابق العلوي. عندما وصلت إلى الطابق، شعرت بالقلق لأنني لم أكن أعرف الطريق. ولكن، بفضل الدعم من室友ي